# Mission 4: Mini-Google & Name Generator 🔎

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/04_mini_google.ipynb)

> **From your syllabus**
> - *Applied Probability*, Unit IV (Markov chains: Chapman-Kolmogorov, limiting probabilities) and Unit II (law of large numbers)
> - *Linear Algebra*: eigenvectors

**Your mission:** rank web pages the way Google once did, then make a machine invent new names. Both are the same object: a **Markov chain**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
pages = ["Home", "Maths", "ML", "Python", "Blog", "Jobs"]
# A[i, j] = 1 means page i links to page j
A = np.array([
    [0, 1, 1, 1, 0, 0],
    [1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 0, 1, 0, 1, 0],
    [1, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 0],
], dtype=float)
print("links out of each page:", A.sum(axis=1))

## 1. The random surfer
A surfer sits on a page and clicks a random outgoing link. That is a Markov chain with transition matrix
$P_{ij}=A_{ij}/\sum_kA_{ik}$ (each row sums to 1). After $n$ clicks the distribution is $\pi_n=\pi_0P^n$ (**Chapman-Kolmogorov**).

**Core 1:** build $P$ from $A$.

<details><summary>Hint</summary>

Divide each row by its sum: `A / A.sum(axis=1, keepdims=True)`.
</details>

In [ ]:
P = ____                           # TODO: row-normalise A
print(np.round(P, 2))
print("rows sum to 1:", np.allclose(P.sum(axis=1), 1))

## 2. Where does the surfer spend time?
The **limiting distribution** $\pi$ satisfies $\pi=\pi P$ (a left eigenvector with eigenvalue 1). Page importance = long-run fraction of time spent there. To guarantee a unique limit, Google's PageRank adds a **teleport**: with probability $1-\alpha$ jump to a uniformly random page,
$$G=\alpha P+\frac{1-\alpha}{n}\mathbf 1\mathbf 1^\top,\qquad \alpha=0.85 .$$
$G$ has all entries positive, so the chain is irreducible and aperiodic and the limit exists (Applied Probability Unit IV).

**Core 2:** power iteration $\pi\leftarrow\pi G$ starting from the uniform distribution.

In [ ]:
n = len(pages)
alpha = 0.85
G = alpha * P + (1 - alpha) / n * np.ones((n, n))

pi = np.ones(n) / n
for step in range(100):
    pi = ____                      # TODO: one step of the chain, pi times G
print({p: round(float(v), 3) for p, v in zip(pages, pi)})

**Check three ways.** (1) $\pi G=\pi$; (2) $\pi$ is the eigenvector of $G^\top$ for eigenvalue 1; (3) every row of $G^{n}$ converges to $\pi$ (Chapman-Kolmogorov).

In [ ]:
print("1. pi G = pi            :", np.allclose(pi @ G, pi))

vals, vecs = np.linalg.eig(G.T)
v = np.real(vecs[:, np.argmax(np.real(vals))]); v = v / v.sum()
print("2. eigenvector matches  :", np.allclose(v, pi, atol=1e-6), " (eigenvalue", round(np.real(vals).max(), 6), ")")

G50 = np.linalg.matrix_power(G, 50)
print("3. rows of G^50 equal pi:", np.allclose(G50, np.tile(pi, (n, 1)), atol=1e-6))

order = np.argsort(-pi)
plt.bar([pages[i] for i in order], pi[order]); plt.ylabel("PageRank"); plt.show()

## 3. The law of large numbers, live
Simulate one surfer for many clicks. The **fraction of time** on each page should approach $\pi$ (the ergodic theorem, a cousin of the LLN from Applied Probability Unit II).

In [ ]:
steps = 20000
state = 0
visits = np.zeros(n)
for _ in range(steps):
    visits[state] += 1
    state = np.random.choice(n, p=G[state])

print("simulated:", np.round(visits / steps, 3))
print("exact pi :", np.round(pi, 3))
print("max error:", round(np.abs(visits / steps - pi).max(), 4))

**Challenge:** make *Jobs* link to *Blog* only (edit row 5 of `A`) and re-run. Which page loses rank, which gains? Explain using the chain.

## 4. Same maths, new game: a name generator
Treat letters as states. `^` marks the start of a name and `$` the end. Count how often letter $a$ is followed by letter $b$, normalise each row, and you have a Markov chain over letters. Sampling from it **invents** names.

In [ ]:
names = """
aadhya aarav abhinav adithya ajay akhil akshay amal amrita anand anjali anoop anusha arjun arun aswin athira
bhavya devika dhanya dileep divya gayathri gopika hari haritha indu irfan jishnu kavya kiran krishna lakshmi
mahesh meera midhun nandana neha nikhil nimisha pooja prakash priya rahul rohit sachin sanjay sarika shiva
sneha sreya suresh swathi tara varun vignesh vishnu yamuna zahra
""".split()
print(len(names), "training names, e.g.", names[:5])

chars = ["^"] + sorted(set("".join(names))) + ["$"]
idx = {c: i for i, c in enumerate(chars)}
counts = np.zeros((len(chars), len(chars))) + 0.01      # tiny smoothing so no row is all zeros
for name in names:
    seq = ["^"] + list(name) + ["$"]
    for a, b in zip(seq[:-1], seq[1:]):
        counts[idx[a], idx[b]] += 1
print("transition counts:", counts.shape)

**Core 3:** turn counts into transition probabilities (same row-normalising idea as before).

In [ ]:
T = ____                           # TODO: row-normalise counts
assert np.allclose(T.sum(axis=1), 1)

def invent(rng):
    state, out = idx["^"], ""
    while True:
        state = rng.choice(len(chars), p=T[state])
        if chars[state] == "$" or len(out) > 14:
            return out
        out += chars[state]

rng = np.random.RandomState(7)
print([invent(rng) for _ in range(12)])

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(T, cmap="Blues"); plt.xticks(range(len(chars)), chars); plt.yticks(range(len(chars)), chars)
plt.xlabel("next letter"); plt.ylabel("current letter"); plt.title("letter transition matrix"); plt.show()

## Stretch
Use **two** previous letters as the state (a second-order chain) and compare the names. This is exactly the step from a bigram model toward a language model: more context, better text, but many more states.

## Quest (take-home)
Replace the list with 50 names from your own language or family tradition (with permission), generate 20 invented names and pick your favourite. Then compute the stationary distribution of letters and report the three most common letters.

## You just used...
- **Markov chains**, the transition matrix and **Chapman-Kolmogorov** $P^{(n)}=P^n$ (Applied Probability Unit IV)
- **limiting probabilities** as an eigenvector, and why teleportation makes them exist
- the **law of large numbers** for long-run averages (Unit II)
- a **language model in miniature**: predict the next token from the previous one

## Your CV line
> *Implemented PageRank via power iteration and eigen-decomposition and built a character-level Markov chain text generator in NumPy; verified convergence to the stationary distribution by simulation.*